In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Ingest WITS Tariff (Production)
# MAGIC 
# MAGIC **MODE = incremental** (default, scheduled):
# MAGIC - Skips if last run was < 90 days ago (annual data, quarterly check is sufficient)
# MAGIC - Deletes old data file after writing new one (keeps storage flat)
# MAGIC 
# MAGIC **MODE = backfill** (manual):
# MAGIC - Ignores skip logic — always runs
# MAGIC - Deletes old data files (clean slate)
# MAGIC - Same API call (WITS has no date-range filter — always returns all years)

# COMMAND ----------

import json
import time
import datetime as dt
from typing import Optional, Dict, Any
import requests

# COMMAND ----------

# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "wits_tariff_raw"

REPORTER = "CAN"
PARTNER  = "WLD"
PRODUCT  = "ALL"
INDICATOR = "AHS-WGHTD-AVRG"
MIN_DAYS_BETWEEN_RUNS = 90

BASE_URL = (
    "https://wits.worldbank.org/API/V1/SDMX/V21/datasource/tradestats-tariff"
    f"/reporter/{REPORTER}/year/ALL/partner/{PARTNER}/product/{PRODUCT}/indicator/{INDICATOR}"
)

# COMMAND ----------

# ---------------- Mode ----------------
dbutils.widgets.dropdown("MODE", "incremental", ["incremental", "backfill"])
MODE = dbutils.widgets.get("MODE").strip().lower()
print(f"MODE: {MODE}")

# COMMAND ----------

# ---------------- Helpers ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def get_json_with_retry(url: str, retries: int = 5, timeout: int = 30) -> Dict[str, Any]:
    backoff = 2
    last_err = None
    headers = {"User-Agent": "databricks-lakehouse-demo/1.0"}
    for i in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if i == retries - 1:
                break
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

# COMMAND ----------

# ---------------- Skip check (incremental only) ----------------
if MODE == "incremental":
    last_run_df = spark.sql("""
        SELECT MAX(last_run_ts) AS last_ts
        FROM canada_business.bronze.ingest_watermarks
        WHERE source = 'wits'
    """).collect()

    last_ts = last_run_df[0]["last_ts"]
    if last_ts is not None:
        last_run_date = dt.datetime.strptime(last_ts[:8], "%Y%m%d").date()
        days_since = (dt.date.today() - last_run_date).days
        if days_since < MIN_DAYS_BETWEEN_RUNS:
            print(f"⏭️ Last run was {days_since} days ago (< {MIN_DAYS_BETWEEN_RUNS}). Skipping.")
            dbutils.notebook.exit(f"SKIPPED: last run {days_since} days ago")
        else:
            print(f"Last run was {days_since} days ago. Proceeding.")
    else:
        print("No previous run found. Running initial ingest.")
else:
    print("Backfill mode — skipping recency check.")

# COMMAND ----------

# ---------------- Resolve paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)
data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

# COMMAND ----------

# ---------------- Fetch + Write ----------------
ts = run_ts_utc()
url = BASE_URL + "?format=JSON"

print(f"⏳ Fetching WITS tariff data... ({MODE})")
payload = get_json_with_retry(url)

new_filename = f"wits_tariff_{REPORTER}_{INDICATOR}_{ts}.json"
out_file = join_uri(data_base, new_filename)
dbutils.fs.put(out_file, json.dumps(payload, indent=2), overwrite=False)
print(f"✅ Written: {out_file}")

# Delete old data files (keep only the new one)
try:
    for f in dbutils.fs.ls(data_base):
        if f.name != new_filename and f.name.endswith(".json"):
            dbutils.fs.rm(f.path)
            print(f"🗑️ Deleted: {f.name}")
except Exception as e:
    print(f"⚠️ Cleanup warning: {e}")

# Write manifest
manifest = {
    "run_ts": ts,
    "mode": MODE,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "bronze_base": bronze_base,
    "source": "WITS API (TradeStats-Tariff SDMX)",
    "request": {
        "reporter": REPORTER, "partner": PARTNER,
        "product": PRODUCT, "indicator": INDICATOR, "url": url
    },
    "files": [{"raw_file": out_file}]
}

manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

# Update watermark
spark.sql(f"""
    MERGE INTO canada_business.bronze.ingest_watermarks AS t
    USING (SELECT
        'wits' AS source, 'ALL' AS series_key,
        current_date() AS last_obs_date, '{ts}' AS last_run_ts,
        1 AS rows_written, current_timestamp() AS updated_at
    ) AS s
    ON t.source = s.source AND t.series_key = s.series_key
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"\n✅ WITS tariff ingest complete ({MODE})")
print("Raw file:", out_file)
print("Manifest:", manifest_path)